# Data Preprocessing
Preprocess `cross_references.tsv` according to the initial EDA.

In [1]:
import pandas as pd

In [2]:
raw_data = pd.read_csv('cross_references.tsv', sep='\t')
raw_data.sample(10)

,From Verse,To Verse,Votes,#www.openbible.info CC-BY 2026-09-07
237665,Matt.16.1,Acts.4.1,5,NaN
199331,Ezek.17.24,Ezek.24.14,4,NaN
71792,1Kgs.14.21,Ps.78.68-Ps.78.69,2,NaN
256613,Luke.11.34,2Kgs.6.15-2Kgs.6.20,2,NaN
259049,Luke.16.24,Rev.20.15,5,NaN
177170,Jer.7.15,Hos.1.4,2,NaN
49722,Josh.13.26,1Kgs.22.3,2,NaN
323697,Titus.3.3,Prov.1.22-Prov.1.23,4,NaN
182285,Jer.23.5,Isa.4.2,23,NaN
67875,1Kgs.2.4,2Sam.7.11-2Sam.7.16,2,NaN


In [3]:
data = raw_data.iloc[:,0:3]
display(data.shape)
display(data.sample(5))

(344756, 3)

,From Verse,To Verse,Votes
125489,Ps.78.58,Deut.32.21,7
116369,Ps.36.2,Ps.10.3,4
201324,Ezek.22.16,Ps.9.16,2
150009,Eccl.3.4,Luke.6.21-Luke.6.25,8
58215,1Sam.11.5,Isa.22.1,2


In [4]:
# remove negative and zero votes
data_positive = data.loc[data['Votes']>0]
display(data_positive.shape)
display(data_positive.sample(5))

(341242, 3)

,From Verse,To Verse,Votes
250996,Luke.1.49,Jer.10.6,16
26038,Lev.13.4,1Tim.5.24,2
138363,Ps.141.4,Prov.23.6-Prov.23.8,2
207488,Ezek.44.10,Num.18.23,2
84357,1Chr.21.10,Josh.24.15,2


In [5]:
def filter_func(row, max_verse):
    # identify if the from verse and to verse are from the same chapter
    chap1 = '.'.join(row['From Verse'].split('.')[:2])
    chap2 = '.'.join(row['To Verse'].split('.')[:2])
    same_chapter = chap1 == chap2

    # if the To Verse spans multiple verses:
    if '-' in row['To Verse']:
        # Check whether it spans multiple chapters
        start, end = row['To Verse'].split('-')
        start_parts = start.split('.')
        end_parts = end.split('.')

        # check whether 'To Verse' stays within one chapter
        not_multi_chapter = start_parts[:2] == end_parts[:2]
        
        # check the number of verses it spans
        start_verse = int(start_parts[2])
        end_verse = int(end_parts[2])
        verse_span = end_verse - start_verse + 1
    else:
        not_multi_chapter = True
        verse_span = 1
    return same_chapter and not_multi_chapter and (verse_span <= max_verse)

In [10]:
# Apply filter
idx_filter = data_positive.apply(filter_func, max_verse=5, axis=1) # set max verse span to 5
data_filtered = data_positive.loc[idx_filter, :]
display(data_filtered.shape)
display(data_filtered.sample(10))

(14792, 3)

,From Verse,To Verse,Votes
206259,Ezek.38.16,Ezek.38.8-Ezek.38.9,3
97948,Neh.13.11,Neh.13.17,4
290251,Rom.5.1,Rom.5.9-Rom.5.10,13
189787,Jer.48.3,Jer.48.34,4
307623,Gal.5.12,Gal.5.10,5
135098,Ps.119.81,Ps.119.20,1
204292,Ezek.33.2,Ezek.33.7,2
151797,Eccl.10.10,Eccl.10.15,9
250212,Mark.16.1,Mark.16.8,4
271059,John.14.7,John.14.16-John.14.20,7


In [11]:
# save filtered data
data_filtered.to_csv('data_filtered.tsv', sep='\t')